In [2]:
import geopandas as gpd
import fsspec
import pyarrow.parquet as pq
import fsspec
from shapely.geometry import box

In [3]:
GADM_L0 = 'gs://crs_climate_data_public/GADM/GADM/gadm410/gadm_410levels__adm_0.gpkg'
GADM_L1 = 'gs://crs_climate_data_public/GADM/GADM/gadm410/gadm_410levels__adm_1.gpkg'
COASTLINE = 'gs://crs_climate_data_public/GADM/GADM/Coastline/hyd_glo_msk_30s.tif'

In [4]:
# Read only the geometry column first to filter
dataset = pq.ParquetDataset("gs://crs_climate_data_public/GADM/parquet/adm0.parquet")
geometry_table = dataset.read(columns=['geometry'])

# Convert to GeoSeries for spatial filtering
gdf_geoms = gpd.GeoSeries.from_arrow(geometry_table['geometry'])
bbox = box(-10, 35, 40, 70)

# Get indices that intersect bbox
mask = gdf_geoms.intersects(bbox)
indices = mask[mask].index

# Now read only the relevant rows
if len(indices) > 0:
    # Need to convert indices to row filters - this is complex
    # Alternative: read with filter on row groups if available
    pass

AttributeError: 'pyarrow.lib.ChunkedArray' object has no attribute '__arrow_c_array__'

# Schema

In [2]:
# For local file or with fsspec
with fsspec.open("gs://crs_climate_data_public/GADM/parquet/adm1.parquet", anonymous=True) as f:
    # Read only the metadata
    parquet_file = pq.ParquetFile(f)
    
    # Get schema information
    schema = parquet_file.schema
    metadata = parquet_file.metadata
    
    print("Schema:")
    print(schema)
    
    print("\nColumn names:", schema.names)
    print("\nColumn types:", [schema.column(i).physical_type for i in range(len(schema.names))])
    
    # Get row group information
    print(f"\nNumber of row groups: {metadata.num_row_groups}")
    print(f"Total rows: {metadata.num_rows}")
    
    # For GeoParquet, check geo metadata
    if b'geo' in metadata.metadata:
        import json
        geo_meta = json.loads(metadata.metadata[b'geo'].decode('utf-8'))
        print("\nGeoParquet metadata:", geo_meta)

Schema:
required group field_id=-1 schema {
  optional binary field_id=-1 GID_0 (String);
  optional binary field_id=-1 COUNTRY (String);
  optional binary field_id=-1 geometry;
}


Column names: ['GID_0', 'COUNTRY', 'geometry']

Column types: ['BYTE_ARRAY', 'BYTE_ARRAY', 'BYTE_ARRAY']

Number of row groups: 1
Total rows: 263

GeoParquet metadata: {'primary_column': 'geometry', 'columns': {'geometry': {'encoding': 'WKB', 'crs': {'$schema': 'https://proj.org/schemas/v0.7/projjson.schema.json', 'type': 'GeographicCRS', 'name': 'WGS 84', 'datum_ensemble': {'name': 'World Geodetic System 1984 ensemble', 'members': [{'name': 'World Geodetic System 1984 (Transit)'}, {'name': 'World Geodetic System 1984 (G730)'}, {'name': 'World Geodetic System 1984 (G873)'}, {'name': 'World Geodetic System 1984 (G1150)'}, {'name': 'World Geodetic System 1984 (G1674)'}, {'name': 'World Geodetic System 1984 (G1762)'}, {'name': 'World Geodetic System 1984 (G2139)'}], 'ellipsoid': {'name': 'WGS 84', 'semi_major_